# Label-Flipping Data Poisoning on Federated FEMNIST

This notebook holds the whole project in one place: data, model, federated training, the
label-flipping attack, its effect, the defense, and the extra tests. It follows the setup of
Tolpegin et al., *Data Poisoning Attacks Against Federated Learning Systems* (ESORICS 2020),
on LEAF FEMNIST (62 classes of handwritten characters).

**Flow**
1. Setup
2. Load FEMNIST and look at the data
3. Train / test loaders
4. Model
5. Federated learning functions: split, local training, FedAvg, evaluation
6. The attack: label flipping
7. `run_experiment()`: one complete federated run
8. Baseline (no attack)
9. Attack runs: `O` → `0` and `A` → `h`
10. All attack levels (saved runs)
11. Attack analysis
12. Defense (Algorithm 1)
13. Prediction confidence and sample analysis
14. Easy, hard and self-made test data
15. Speed: latency and throughput
16. Summary

Sections 8–9 train live (about 25 minutes per run on one CPU thread). Section 10 trains about
60 runs, so by default it loads the saved results of those runs. Set `RUN_FULL_GRID = True` to
re-run them instead. Every run is seeded by its id, so a re-run reproduces the saved numbers
exactly; sections 8–9 check this.

## 1. Setup

In [ ]:
import os
import glob
import json
import copy
import math
import time
import random
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, silhouette_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# run from the repository root, also when the notebook is opened from notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# CPU on purpose: the saved runs were made on CPU with 1 thread per run, and a GPU or a
# different thread count changes the last digits of the results.
device = torch.device("cpu")
NUM_THREADS = 1
torch.set_num_threads(NUM_THREADS)

LEAF_FEMNIST_PATH = "data/leaf_femnist"   # output of LEAF's preprocess.sh (train/ and test/)
RESULTS_DIR = "results/label_flipping"     # saved runs of the full grid (section 10)
DEFENSE_DIR = "results_defense/label_flipping"
MODELS_DIR = "results_models/label_flipping"
OUTPUT_DIR = "notebook_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# experiment settings (the paper's Fashion-MNIST settings, kept for FEMNIST)
NUM_CLIENTS = 50          # N
CLIENTS_PER_ROUND = 5     # k
NUM_ROUNDS = 200          # R
BATCH_SIZE = 4
TEST_BATCH_SIZE = 1000
LEARNING_RATE = 0.001
MOMENTUM = 0.9
LR_STEP = 10              # x0.1 every 10 local epochs of a participant
LR_GAMMA = 0.1
MIN_LR = 1e-10
NUM_CLASSES = 62

LABELS = [str(i) for i in range(10)] + [chr(65 + i) for i in range(26)] + [chr(97 + i) for i in range(26)]
O, ZERO, A, H = LABELS.index("O"), LABELS.index("0"), LABELS.index("A"), LABELS.index("h")

# colors used in every plot
C_CLEAN, C_O, C_A, C_GRAY = "#1F8A70", "#D9480F", "#2E6DB4", "#8A94A6"

print("PyTorch:", torch.__version__)
print("Device:", device, "| threads:", NUM_THREADS)
print("Classes:", NUM_CLASSES, "| 'O' =", O, "| '0' =", ZERO, "| 'A' =", A, "| 'h' =", H)

## 2. Load FEMNIST and look at the data

The data is LEAF's FEMNIST: NIST Special Database 19, prepared by the LEAF benchmark
(https://github.com/TalwalkarLab/leaf) with
`./preprocess.sh -s niid --sf 0.1 -k 0 -t sample --tf 0.9`. That is a 10% sample of the
writers, with each writer's images split 90/10 into train and test. Each sample is 784 pixel
values (28×28, 1.0 = white) and a label from 0 to 61.

In [ ]:
def load_leaf_json_dir(directory):
    """Read every LEAF json file in a folder and pool all writers."""
    files = sorted(glob.glob(os.path.join(directory, "*.json")))
    xs, ys, writers = [], [], []
    for path in files:
        with open(path) as f:
            data = json.load(f)
        for user in data["users"]:
            xs.append(np.asarray(data["user_data"][user]["x"], dtype=np.float32))
            ys.append(np.asarray(data["user_data"][user]["y"], dtype=np.int64))
            writers.extend([user] * len(data["user_data"][user]["y"]))
    X = np.concatenate(xs).reshape(-1, 1, 28, 28)
    Y = np.concatenate(ys)
    return X, Y, np.asarray(writers)


X_train_raw, Y_train_raw, writers_train = load_leaf_json_dir(os.path.join(LEAF_FEMNIST_PATH, "train"))
X_test_raw, Y_test_raw, writers_test = load_leaf_json_dir(os.path.join(LEAF_FEMNIST_PATH, "test"))

print("Train images:", len(Y_train_raw), "| Test images:", len(Y_test_raw))
print("Writers:", len(set(writers_train)))
print("Image shape:", X_train_raw.shape[1:], "| pixel range:", X_train_raw.min(), "to", X_train_raw.max())
print("Train : test = {:.1f} : {:.1f}".format(100 * len(Y_train_raw) / (len(Y_train_raw) + len(Y_test_raw)),
                                              100 * len(Y_test_raw) / (len(Y_train_raw) + len(Y_test_raw))))

In [ ]:
# check one sample
print("Writer:", writers_train[0], "| label:", Y_train_raw[0], "=", repr(LABELS[Y_train_raw[0]]))
print("First 10 classes:", LABELS[:10])

In [ ]:
# images per class
class_counts = pd.DataFrame({
    "class": LABELS,
    "train": np.bincount(Y_train_raw, minlength=NUM_CLASSES),
    "test": np.bincount(Y_test_raw, minlength=NUM_CLASSES),
})
for name, part in [("digits", class_counts[:10]), ("uppercase", class_counts[10:36]), ("lowercase", class_counts[36:])]:
    print("{:10s} train {:6d} ({}-{} per class) | test {:5d} ({}-{} per class)".format(
        name, part.train.sum(), part.train.min(), part.train.max(), part.test.sum(), part.test.min(), part.test.max()))
print(class_counts[class_counts["class"].isin(["0", "O", "A", "h"])].to_string(index=False))

plt.figure(figsize=(14, 4))
colors = [C_O if c in ("O", "0") else C_A if c in ("A", "h") else C_GRAY for c in LABELS]
plt.bar(range(NUM_CLASSES), class_counts["train"], color=colors)
plt.xticks(range(NUM_CLASSES), LABELS)
plt.ylabel("Training images")
plt.title("FEMNIST training images per class (attacked pairs in color)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# a few training examples
examples = "0O1lAh5Sek"
fig, axes = plt.subplots(1, len(examples), figsize=(14, 2))
for ax, ch in zip(axes, examples):
    idx = np.where(Y_train_raw == LABELS.index(ch))[0][7]
    ax.imshow(X_train_raw[idx, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title("'{}' (#{})".format(ch, LABELS.index(ch)))
    ax.axis("off")
plt.suptitle("Training examples")
plt.show()

## 3. Train / test loaders

The pooled images are shuffled once with seed 1 and stored in a train loader (batch 4) and a
test loader (batch 1000). Every experiment starts from these same loaders. If the repo's saved
loaders (`data_loaders/femnist/*.pickle`) are present, the last cell checks that they match.

In [ ]:
def shuffle_data(X, Y):
    data = list(zip(X, Y))
    random.shuffle(data)
    X, Y = zip(*data)
    return np.asarray(X), np.asarray(Y)


def make_loader(X, Y, batch_size, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(Y).long())
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


random.seed(1)
X_train, Y_train = shuffle_data(X_train_raw, Y_train_raw)
X_test, Y_test = shuffle_data(X_test_raw, Y_test_raw)

train_loader = make_loader(X_train, Y_train, BATCH_SIZE)
test_loader = make_loader(X_test, Y_test, TEST_BATCH_SIZE)
X_test_t, Y_test_t = test_loader.dataset.tensors

print("Train batches:", len(train_loader), "| Test batches:", len(test_loader))

In [ ]:
saved = "data_loaders/femnist/train_data_loader.pickle"
if os.path.exists(saved):
    with open(saved, "rb") as f:
        saved_train = pickle.load(f).dataset.tensors
    with open(saved.replace("train", "test"), "rb") as f:
        saved_test = pickle.load(f).dataset.tensors
    same = all(torch.equal(a, b) for a, b in zip(saved_train + saved_test, train_loader.dataset.tensors + test_loader.dataset.tensors))
    print("Matches the saved data loaders:", same)

## 4. Model

The paper's Fashion-MNIST CNN, with the output layer widened from 10 to 62 classes. Every
participant and every run starts from the same initial weights θ₀ (seed 1).

In [ ]:
class FEMNISTCNN(nn.Module):

    def __init__(self):
        super(FEMNISTCNN, self).__init__()

        # 1 Convolution layer: 1 -> 16 channels, 28x28 -> 14x14
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2))

        # 2 Convolution layer: 16 -> 32 channels, 14x14 -> 7x7
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2))

        # one output per class; fc.weight[c] are the weights of class c's output node
        self.fc = nn.Linear(7 * 7 * 32, NUM_CLASSES)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


torch.manual_seed(1)
initial_state = copy.deepcopy(FEMNISTCNN().state_dict())

model = FEMNISTCNN()
model.load_state_dict(initial_state)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 5. Federated learning functions

In [ ]:
def split_iid(train_loader, num_clients):
    """Deal the training batches round-robin: batch i goes to participant i % N (IID, equal size)."""
    client_batches = [[] for _ in range(num_clients)]
    for batch_idx, (data, target) in enumerate(train_loader):
        client_batches[batch_idx % num_clients].append((data, target))
    client_data = []
    for batches in client_batches:
        X = np.array([x.numpy() for data, _ in batches for x in data])
        Y = np.array([y.numpy() for _, target in batches for y in target])
        client_data.append((X, Y))
    return client_data


def create_participant(X, Y):
    """A participant keeps its own model, optimizer (with momentum) and learning-rate schedule."""
    model = FEMNISTCNN()
    model.load_state_dict(initial_state)
    return {
        "model": model,
        "optimizer": optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM),
        "local_epochs": 0,
        "loader": make_loader(X, Y, BATCH_SIZE, shuffle=True),
        "size": len(Y),
    }

In [ ]:
criterion = nn.CrossEntropyLoss()


def local_train(participant):
    """One local epoch of SGD on the participant's own data."""
    model, optimizer = participant["model"], participant["optimizer"]
    model.train()

    for images, labels in participant["loader"]:
        images, labels = images.to(device), labels.to(device)

        # initialize gradient
        optimizer.zero_grad()

        # forward, loss, backward, update
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # learning-rate schedule: x0.1 every LR_STEP local epochs
    participant["local_epochs"] += 1
    if participant["local_epochs"] % LR_STEP == 0:
        lr = optimizer.param_groups[0]["lr"]
        if lr * LR_GAMMA >= MIN_LR:
            optimizer.param_groups[0]["lr"] = lr * LR_GAMMA

In [ ]:
def fedavg(client_states):
    """Plain average of the selected participants' parameters (all hold equal data)."""
    new_state = {}
    for key in client_states[0].keys():
        new_state[key] = sum(state[key].data for state in client_states) / len(client_states)
    return new_state

In [ ]:
def evaluate(model, test_loader):
    """Accuracy, summed loss, per-class recall and the confusion matrix on the test set."""
    model.eval()

    correct, total, total_loss = 0, 0, 0.0
    targets, predictions = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            predicted = outputs.argmax(dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            total_loss += criterion(outputs, labels).item()

            targets.extend(labels.numpy())
            predictions.extend(predicted.numpy())

    conf = confusion_matrix(targets, predictions, labels=list(range(NUM_CLASSES)))
    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.nan_to_num(np.diag(conf) / conf.sum(axis=1))

    return {"accuracy": 100 * correct / total, "loss": total_loss, "recall": recall, "confusion": conf}

## 6. The attack: label flipping

A malicious participant relabels every image of the source class as the target class in its
own data, then trains normally. The images themselves are untouched. The server never sees the
data, so it cannot notice.

* **Test 1: `O` → `0`.** Letters the clean model already confuses (frequent pair).
* **Test 2: `A` → `h`.** Letters it almost never confuses (infrequent pair).

In [ ]:
def label_flip(Y, source, target):
    Y = Y.copy()
    Y[Y == source] = target
    return Y


def choose_malicious(num_clients, num_malicious):
    """Random distinct participant ids (uses Python's random, seeded per experiment)."""
    ids = []
    while len(ids) < num_malicious:
        r = random.randint(0, num_clients - 1)
        if r not in ids:
            ids.append(r)
    return ids


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)

In [ ]:
# what the attack does to one participant's data (example: 10% malicious, seed 3000)
seed_everything(3000)
example_data = split_iid(train_loader, NUM_CLIENTS)
example_malicious = choose_malicious(NUM_CLIENTS, 5)
print("Malicious participants:", example_malicious)

for client_id in [0, example_malicious[0]]:
    X, Y = example_data[client_id]
    if client_id in example_malicious:
        Y = label_flip(Y, O, ZERO)
        kind = "MALICIOUS"
    else:
        kind = "BENIGN"
    print("Client {:2d}: {:9s} | images {} | '0': {:3d} | 'O': {:3d}".format(
        client_id, kind, len(Y), (Y == ZERO).sum(), (Y == O).sum()))

## 7. `run_experiment()`: one complete federated run

Each round:
1. pick 5 of the 50 participants at random;
2. each trains one local epoch, malicious ones on flipped labels;
3. the server averages the 5 models with FedAvg;
4. every participant receives the new global model;
5. the global model is evaluated on the test set.

With `record_deltas=True` it also keeps each update's change to the source class's output
weights, θ_local − θ_global (1,568 numbers), which the defense needs (section 12).

In [ ]:
def run_experiment(source, target, malicious_pct, seed=3000, num_rounds=NUM_ROUNDS,
                   record_deltas=False, print_every=25):
    seed_everything(seed)

    client_data = split_iid(train_loader, NUM_CLIENTS)
    num_malicious = int(round(NUM_CLIENTS * malicious_pct / 100.0))
    malicious = choose_malicious(NUM_CLIENTS, num_malicious)
    client_data = [(X, label_flip(Y, source, target)) if i in malicious else (X, Y)
                   for i, (X, Y) in enumerate(client_data)]
    participants = [create_participant(X, Y) for X, Y in client_data]

    history = {"round": [], "accuracy": [], "loss": [], "source_recall": [], "target_recall": [],
               "selected": [], "malicious_in_round": [], "seconds": []}
    recall_per_round = []
    deltas = {"round": [], "participant": [], "malicious": [], "delta": []}

    for round_num in range(1, num_rounds + 1):
        t0 = time.perf_counter()
        selected = random.sample(range(NUM_CLIENTS), CLIENTS_PER_ROUND)

        if record_deltas:
            global_row = participants[0]["model"].state_dict()["fc.weight"][source].clone()

        for i in selected:
            local_train(participants[i])
            if record_deltas:
                local_row = participants[i]["model"].state_dict()["fc.weight"][source]
                deltas["round"].append(round_num)
                deltas["participant"].append(i)
                deltas["malicious"].append(i in malicious)
                deltas["delta"].append((local_row - global_row).numpy().copy())

        new_state = fedavg([participants[i]["model"].state_dict() for i in selected])
        for p in participants:
            p["model"].load_state_dict(copy.deepcopy(new_state), strict=True)

        result = evaluate(participants[0]["model"], test_loader)

        history["round"].append(round_num)
        history["accuracy"].append(result["accuracy"])
        history["loss"].append(result["loss"])
        history["source_recall"].append(100 * result["recall"][source])
        history["target_recall"].append(100 * result["recall"][target])
        history["selected"].append(selected)
        history["malicious_in_round"].append(sum(i in malicious for i in selected))
        history["seconds"].append(time.perf_counter() - t0)
        recall_per_round.append(result["recall"])

        if round_num % print_every == 0 or round_num == num_rounds:
            print("Round {:3d}/{} | accuracy {:6.2f}% | '{}' recall {:6.2f}% | attackers this round {}".format(
                round_num, num_rounds, result["accuracy"], LABELS[source], 100 * result["recall"][source],
                history["malicious_in_round"][-1]))

    final_model = FEMNISTCNN()
    final_model.load_state_dict(participants[0]["model"].state_dict())
    final_model.eval()
    return {
        "source": source, "target": target, "malicious_pct": malicious_pct, "seed": seed,
        "malicious": sorted(malicious), "history": history, "recall_per_round": np.array(recall_per_round),
        "confusion": result["confusion"], "model": final_model,
        "deltas": {k: np.array(v) for k, v in deltas.items()} if record_deltas else None,
    }


def saved_results_path(source, target, malicious_pct, seed=3000):
    pair = "0_to_0" if malicious_pct == 0 else "{}_to_{}".format(source, target)
    return os.path.join(RESULTS_DIR, pair, "m{}".format(malicious_pct), "{}_results.csv".format(seed))


def check_against_saved(run):
    """Compare a live run with the saved run of the same id (columns: accuracy, loss)."""
    path = saved_results_path(run["source"], run["target"], run["malicious_pct"], run["seed"])
    if not os.path.exists(path):
        print("No saved run to compare with:", path)
        return
    saved = pd.read_csv(path, header=None)
    live = np.array([run["history"]["accuracy"], run["history"]["loss"]]).T
    n = len(live)
    same = np.allclose(saved.values[:n, :2], live, rtol=0, atol=1e-9)
    print("Identical to the saved run ({}) for all {} rounds: {}".format(path, n, same))

## 8. Baseline: no attack

In [ ]:
baseline = run_experiment(source=0, target=0, malicious_pct=0, seed=3000)
check_against_saved(baseline)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(baseline["history"]["round"], baseline["history"]["accuracy"], color=C_CLEAN, label="Global accuracy")
plt.plot(baseline["history"]["round"], baseline["recall_per_round"][:, O] * 100, color=C_O, alpha=0.8, label="'O' recall")
plt.plot(baseline["history"]["round"], baseline["recall_per_round"][:, A] * 100, color=C_A, alpha=0.8, label="'A' recall")
plt.xlabel("Communication round")
plt.ylabel("%")
plt.ylim(0, 100)
plt.title("Baseline (no attack)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
baseline_df = pd.DataFrame({k: v for k, v in baseline["history"].items() if k not in ("selected",)})
baseline_df.to_csv(os.path.join(OUTPUT_DIR, "baseline_history.csv"), index=False)
print(baseline_df.tail(5).to_string(index=False))

## 9. Attack runs: `O` → `0` and `A` → `h` (20% malicious)

Same seed as the baseline, so the data split and the participant picks are identical. The
only difference is that 10 of the 50 participants flip their labels. Both runs record their
updates for the defense.

In [ ]:
attack_O = run_experiment(source=O, target=ZERO, malicious_pct=20, seed=3000, record_deltas=True)
check_against_saved(attack_O)

In [ ]:
attack_A = run_experiment(source=A, target=H, malicious_pct=20, seed=3000, record_deltas=True)
check_against_saved(attack_A)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, run, src, col in [(axes[0], attack_O, O, C_O), (axes[1], attack_A, A, C_A)]:
    rounds = run["history"]["round"]
    ax.plot(rounds, baseline["recall_per_round"][:, src] * 100, color=C_GRAY, label="'{}' recall, no attack".format(LABELS[src]))
    ax.plot(rounds, run["history"]["source_recall"], color=col, label="'{}' recall, 20% malicious".format(LABELS[src]))
    ax.plot(rounds, run["history"]["accuracy"], color=C_CLEAN, label="Global accuracy, 20% malicious")
    ax.set_title("'{}' -> '{}'".format(LABELS[run["source"]], LABELS[run["target"]]))
    ax.set_xlabel("Communication round")
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")
axes[0].set_ylabel("%")
plt.suptitle("Accuracy holds while the attacked letter drops and swings")
plt.tight_layout()
plt.show()

In [ ]:
summary_live = pd.DataFrame([
    {"run": name, "accuracy": r["history"]["accuracy"][-1],
     "'O' recall": 100 * r["recall_per_round"][-1, O], "'A' recall": 100 * r["recall_per_round"][-1, A],
     "malicious ids": r["malicious"] if r["malicious"] else "-"}
    for name, r in [("baseline", baseline), ("O->0, 20%", attack_O), ("A->h, 20%", attack_A)]
])
print(summary_live.round(2).to_string(index=False))

## 10. All attack levels (saved runs)

The full grid has 8 levels × 2 pairs, with 3 repetitions each (10 for the baseline), and each
run takes about 25 minutes on one CPU thread. By default this section loads the saved results.
With `RUN_FULL_GRID = True`, it runs them with `run_experiment()` and saves them in the same
layout.

In [ ]:
RUN_FULL_GRID = False
MALICIOUS_PCTS = [0, 2, 4, 10, 20, 30, 40, 50]
PAIRS = {"O->0": (O, ZERO), "A->h": (A, H)}


def seeds_for(pct):
    return range(3000, 3010) if pct == 0 else range(3000, 3003)


if RUN_FULL_GRID:
    for name, (src, tgt) in PAIRS.items():
        for pct in MALICIOUS_PCTS:
            for seed in seeds_for(pct):
                path = saved_results_path(src, tgt, pct, seed)
                if os.path.exists(path):
                    continue
                run = run_experiment(src, tgt, pct, seed=seed, print_every=200)
                os.makedirs(os.path.dirname(path), exist_ok=True)
                rows = np.column_stack([run["history"]["accuracy"], run["history"]["loss"],
                                        np.zeros((NUM_ROUNDS, NUM_CLASSES)), run["recall_per_round"]])
                np.savetxt(path, rows, delimiter=",")
                np.save(path.replace("_results.csv", "_final_confusion.npy"), run["confusion"])


def load_run(src, tgt, pct, seed):
    """Saved run: per-round accuracy and per-class recall (columns 2+62 onward), final confusion."""
    path = saved_results_path(src, tgt, pct, seed)
    if not os.path.exists(path):
        return None
    values = pd.read_csv(path, header=None).values
    return {"accuracy": values[:, 0], "recall": values[:, 2 + NUM_CLASSES:] * 100,
            "confusion": np.load(path.replace("_results.csv", "_final_confusion.npy"))}


grid = {}
for name, (src, tgt) in PAIRS.items():
    for pct in MALICIOUS_PCTS:
        runs = [load_run(src, tgt, pct, s) for s in seeds_for(pct)]
        grid[(name, pct)] = [r for r in runs if r is not None]

print("Loaded runs per level:", {"{} {}%".format(*k): len(v) for k, v in grid.items()})

In [ ]:
rows = []
for name, (src, tgt) in PAIRS.items():
    base_recall = np.mean([r["recall"][-1, src] for r in grid[(name, 0)]])
    for pct in MALICIOUS_PCTS:
        runs = grid[(name, pct)]
        acc = np.array([r["accuracy"][-1] for r in runs])
        rec = np.array([r["recall"][-1, src] for r in runs])
        rows.append({"pair": name, "malicious %": pct, "runs": len(runs),
                     "accuracy": acc.mean(), "accuracy std": acc.std(),
                     "source recall": rec.mean(), "source recall std": rec.std(),
                     "loss in source recall": base_recall - rec.mean() if pct else np.nan,
                     "target recall": np.mean([r["recall"][-1, tgt] for r in runs])})
grid_df = pd.DataFrame(rows)
grid_df.to_csv(os.path.join(OUTPUT_DIR, "attack_levels.csv"), index=False)
print(grid_df.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(MALICIOUS_PCTS) - 1)
for k, (name, col) in enumerate([("O->0", C_O), ("A->h", C_A)]):
    part = grid_df[(grid_df.pair == name) & (grid_df["malicious %"] > 0)]
    axes[0].bar(x + (k - 0.5) * 0.38, part["loss in source recall"], 0.36, color=col, label=name)
    axes[1].errorbar(MALICIOUS_PCTS, grid_df[grid_df.pair == name]["accuracy"],
                     yerr=grid_df[grid_df.pair == name]["accuracy std"], color=col, marker="o", capsize=3, label=name)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MALICIOUS_PCTS[1:])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xlabel("Malicious participants (%)")
axes[0].set_ylabel("Loss in source-class recall (pts)")
axes[0].set_title("Damage to the attacked letter")
axes[1].set_xlabel("Malicious participants (%)")
axes[1].set_ylabel("Global accuracy (%)")
axes[1].set_title("Global accuracy barely moves")
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Attack analysis

* Where do the source test images go?
* How much does global accuracy drop, and why so little?
* How stable is the attacked letter from round to round?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
where_rows = []
for ax, (name, (src, tgt)), col in zip(axes, PAIRS.items(), [C_O, C_A]):
    correct, to_target, other = [], [], []
    for pct in MALICIOUS_PCTS:
        conf = np.mean([r["confusion"][src] for r in grid[(name, pct)]], axis=0)
        n = conf.sum()
        correct.append(100 * conf[src] / n)
        to_target.append(100 * conf[tgt] / n)
        other.append(100 - correct[-1] - to_target[-1])
        where_rows.append({"pair": name, "malicious %": pct, "correct %": correct[-1],
                           "to target %": to_target[-1], "other classes %": other[-1]})
    labels = [str(p) for p in MALICIOUS_PCTS]
    ax.bar(labels, correct, color="#C5CCD8", label="correct")
    ax.bar(labels, to_target, bottom=correct, color=col, label="labeled '{}'".format(LABELS[tgt]))
    ax.bar(labels, other, bottom=np.array(correct) + np.array(to_target), color="#3B4252", label="other classes")
    ax.set_title("Where the '{}' test images go".format(LABELS[src]))
    ax.set_xlabel("Malicious participants (%)")
    ax.set_ylim(0, 100)
    ax.legend(loc="lower left")
axes[0].set_ylabel("Share of source test images (%)")
plt.tight_layout()
plt.show()
print(pd.DataFrame(where_rows).round(1).to_string(index=False))

In [ ]:
# accuracy drop ~= source share x source recall loss - target share x target recall gain
test_share = np.bincount(Y_test, minlength=NUM_CLASSES) / len(Y_test)
rule_rows = []
for name, (src, tgt) in PAIRS.items():
    base = grid_df[(grid_df.pair == name) & (grid_df["malicious %"] == 0)].iloc[0]
    worst = grid_df[(grid_df.pair == name) & (grid_df["malicious %"] == 50)].iloc[0]
    predicted = test_share[src] * (base["source recall"] - worst["source recall"]) \
        - test_share[tgt] * (worst["target recall"] - base["target recall"])
    rule_rows.append({"pair": name, "source share of test set %": 100 * test_share[src],
                      "predicted accuracy drop at 50%": predicted,
                      "observed accuracy drop at 50%": base["accuracy"] - worst["accuracy"]})
print(pd.DataFrame(rule_rows).round(2).to_string(index=False))
print("Largest class share of the test set: {:.2f}% ('{}'), so a one-class attack costs at most ~{:.1f} pts".format(
    100 * test_share.max(), LABELS[test_share.argmax()], 100 * test_share.max()))

In [ ]:
# round-to-round stability of the attacked letter over the last 50 rounds
stab_rows = []
for name, (src, tgt) in PAIRS.items():
    for pct in [0, 10, 20, 30, 40, 50]:
        stds = [r["recall"][-50:, src].std() for r in grid[(name, pct)]]
        stab_rows.append({"pair": name, "malicious %": pct, "std of source recall, last 50 rounds": np.mean(stds)})
print(pd.DataFrame(stab_rows).pivot(index="malicious %", columns="pair",
                                    values="std of source recall, last 50 rounds").round(1))

## 12. Defense (Algorithm 1)

The server looks only at the 1,568 weights that feed the source class's output node. For
each update it takes θ_local − θ_global on that row, standardizes it, and projects it onto
2 principal components (the paper's Algorithm 1). Honest participants pull the row toward
the letter's features; attackers push it away, so they separate.

To turn the plot into a decision (our rule, not the paper's):
1. average each participant's PC 1 score over all its updates;
2. split the 50 averages in two with k-means;
3. if the split is clear (silhouette ≥ 0.65), flag the smaller group.

In [ ]:
def defense_points(deltas, first_round=10, last_round=200):
    keep = (deltas["round"] >= first_round) & (deltas["round"] <= last_round)
    X = deltas["delta"][keep]
    points = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
    return points, deltas["malicious"][keep], deltas["participant"][keep]


def participant_rule(pc1, malicious, participants, min_silhouette=0.65):
    ids = sorted(set(participants.tolist()))
    avg = np.array([pc1[participants == p].mean() for p in ids]).reshape(-1, 1)
    is_mal = np.array([malicious[participants == p].any() for p in ids])
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit(avg).labels_
    sil = silhouette_score(avg, labels)
    flagged = (labels == np.argmin(np.bincount(labels))) & (sil >= min_silhouette)
    return {"silhouette": round(float(sil), 3), "acted": bool(sil >= min_silhouette),
            "attackers caught": "{}/{}".format(int((flagged & is_mal).sum()), int(is_mal.sum())),
            "honest flagged": int((flagged & ~is_mal).sum())}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, run, col in [(axes[0], attack_O, C_O), (axes[1], attack_A, C_A)]:
    pts, mal, _ = defense_points(run["deltas"])
    ax.scatter(pts[~mal, 0], pts[~mal, 1], s=10, color=C_GRAY, alpha=0.6, label="honest update")
    ax.scatter(pts[mal, 0], pts[mal, 1], s=25, color=col, marker="x", label="malicious update")
    ax.set_title("'{}' -> '{}', 20% malicious ({} updates)".format(LABELS[run["source"]], LABELS[run["target"]], len(pts)))
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend()
plt.suptitle("Source-class updates after PCA (Algorithm 1)")
plt.tight_layout()
plt.show()

for name, run in [("O->0, 20%", attack_O), ("A->h, 20%", attack_A)]:
    pts, mal, part = defense_points(run["deltas"])
    print(name, participant_rule(pts[:, 0], mal, part))

In [ ]:
# every saved defense run (m = 0, 2, 4, 10, 20%), full window and early window
defense_rows = []
for name, (src, tgt) in PAIRS.items():
    pair_dir = os.path.join(DEFENSE_DIR, "{}_to_{}".format(src, tgt))
    for pct in [0, 2, 4, 10, 20]:
        path = os.path.join(pair_dir, "m{}".format(pct), "3000_deltas.npz")
        if not os.path.exists(path):
            continue
        d = np.load(path)
        deltas = {"round": d["rounds"], "participant": d["workers"], "malicious": d["is_poisoned"],
                  "delta": d["deltas"][:, list(d["classes"]).index(src), :]}
        for window, last in [("rounds 10-200", 200), ("rounds 10-50", 50)]:
            pts, mal, part = defense_points(deltas, 10, last)
            row = {"pair": name, "malicious %": pct, "window": window}
            row.update(participant_rule(pts[:, 0], mal, part))
            defense_rows.append(row)
defense_df = pd.DataFrame(defense_rows)
defense_df.to_csv(os.path.join(OUTPUT_DIR, "defense_detection.csv"), index=False)
print(defense_df.to_string(index=False))

In [ ]:
# why the defense works better for 'A': honest updates on the 'A' row are quieter
size_rows = []
for name, run in [("O->0", attack_O), ("A->h", attack_A)]:
    d = run["deltas"]
    keep = d["round"] >= 10
    norms = np.linalg.norm(d["delta"][keep], axis=1)
    mal = d["malicious"][keep]
    size_rows.append({"pair": name, "honest update size": norms[~mal].mean(),
                      "malicious update size": norms[mal].mean(),
                      "malicious / honest": norms[mal].mean() / norms[~mal].mean()})
print(pd.DataFrame(size_rows).round(4).to_string(index=False))

## 13. Prediction confidence and sample analysis

For each test image we record the predicted class, the confidence of the top-1 class (its
softmax probability), the probability left for the remaining 61 classes, and the probability
given to the attack's target.

In [ ]:
def extract_predictions(model, X, Y, target_label):
    model.eval()
    with torch.no_grad():
        probs = torch.cat([torch.softmax(model(X[i:i + 1000]), dim=1) for i in range(0, len(X), 1000)]).numpy()
    pred = probs.argmax(1)
    return pd.DataFrame({
        "Original_Label": [LABELS[y] for y in Y],
        "Predicted_Label": [LABELS[p] for p in pred],
        "Correct": pred == Y,
        "Top1_Confidence": probs.max(1),
        "Rest_Confidence": 1 - probs.max(1),
        "Target_Confidence": probs[:, target_label],
    }), probs


models = {"clean": baseline["model"], "O->0, 20%": attack_O["model"], "A->h, 20%": attack_A["model"]}
for name, src, tgt in [("O->0, 50%", O, ZERO), ("A->h, 50%", A, H)]:
    path = os.path.join(MODELS_DIR, "{}_to_{}".format(src, tgt), "m50", "3000_final_model.pt")
    if os.path.exists(path):
        m = FEMNISTCNN()
        m.load_state_dict(torch.load(path, map_location="cpu"))
        models[name] = m.eval()

Y_test_np = Y_test_t.numpy()
conf_rows = []
predictions = {}
for name, m in models.items():
    df, probs = extract_predictions(m, X_test_t, Y_test_np, ZERO if name.startswith("O") else H)
    predictions[name] = (df, probs)
    conf_rows.append({"model": name, "accuracy": 100 * df.Correct.mean(),
                      "top-1 confidence": df.Top1_Confidence.mean(),
                      "top-1 conf. when right": df[df.Correct].Top1_Confidence.mean(),
                      "top-1 conf. when wrong": df[~df.Correct].Top1_Confidence.mean(),
                      "mass on other 61 classes": df.Rest_Confidence.mean(),
                      "'O' recall": 100 * (df[Y_test_np == O].Predicted_Label == "O").mean(),
                      "'A' recall": 100 * (df[Y_test_np == A].Predicted_Label == "A").mean()})
confidence_df = pd.DataFrame(conf_rows)
print(confidence_df.round(3).to_string(index=False))

In [ ]:
# one 'O' test image: top-1 class, next classes, and the rest
idx = int(np.where(Y_test_np == O)[0][0])
fig, axes = plt.subplots(1, len(models) + 1, figsize=(4 * (len(models) + 1), 3.2))
axes[0].imshow(X_test_t[idx, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("true 'O' (test #{})".format(idx))
axes[0].axis("off")
for ax, (name, (df, probs)) in zip(axes[1:], predictions.items()):
    p = probs[idx]
    top = np.argsort(-p)[:4]
    values = list(p[top]) + [1 - p[top].sum()]
    names = ["'{}'".format(LABELS[t]) for t in top] + ["other 58"]
    ax.barh(names[::-1], values[::-1], color=[C_GRAY] * 4 + [C_O if name.startswith("O") else C_A if name.startswith("A") else C_CLEAN])
    for y, v in enumerate(values[::-1]):
        ax.text(v + 0.02, y, "{:.1%}".format(v), va="center")
    ax.set_xlim(0, 1.25)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# compare clean vs attacked prediction for every 'O' test image
attacked_name = "O->0, 50%" if "O->0, 50%" in predictions else "O->0, 20%"
clean_df = predictions["clean"][0][Y_test_np == O].reset_index()
attack_df = predictions[attacked_name][0][Y_test_np == O].reset_index()


def classify_result(clean_ok, attack_ok):
    if clean_ok and attack_ok:
        return "Both correct"
    if clean_ok and not attack_ok:
        return "Flipped by the attack"
    if not clean_ok and attack_ok:
        return "Fixed by the attack"
    return "Both wrong"


sample_comparison = pd.DataFrame({
    "test_index": clean_df["index"],
    "Clean_Prediction": clean_df.Predicted_Label,
    "Attacked_Prediction": attack_df.Predicted_Label,
    "Clean_O_Confidence": predictions["clean"][1][Y_test_np == O][:, O],
    "Attacked_0_Confidence": attack_df.Target_Confidence,
})
sample_comparison["Category"] = [classify_result(a, b) for a, b in zip(clean_df.Correct, attack_df.Correct)]
print("Model compared with clean:", attacked_name)
print(sample_comparison.Category.value_counts().to_string())

In [ ]:
def visualize_category(comparison, category, num_samples=10):
    selected = comparison[comparison.Category == category].head(num_samples)
    if len(selected) == 0:
        print("No samples in category:", category)
        return
    fig, axes = plt.subplots(1, len(selected), figsize=(1.6 * len(selected), 2.2), squeeze=False)
    for ax, (_, row) in zip(axes[0], selected.iterrows()):
        ax.imshow(X_test_t[row.test_index, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title("{}->{}\n{:.0%}".format(row.Clean_Prediction, row.Attacked_Prediction, row.Attacked_0_Confidence), fontsize=9)
        ax.axis("off")
    plt.suptitle("{} (clean -> attacked prediction, confidence in '0')".format(category))
    plt.tight_layout()
    plt.show()


for category in sample_comparison.Category.value_counts().index:
    visualize_category(sample_comparison, category)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(sample_comparison.Clean_O_Confidence, bins=20, alpha=0.6, color=C_CLEAN, label="clean model: confidence in 'O'")
plt.hist(sample_comparison.Attacked_0_Confidence, bins=20, alpha=0.6, color=C_O, label="attacked model: confidence in '0'")
plt.xlabel("Confidence")
plt.ylabel("Number of 'O' test images")
plt.title("The attacked model is confidently wrong")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 14. Easy, hard and self-made test data

* **Easy / hard:** the FEMNIST test set split in advance by visual ambiguity. Hard means
  the class has a look-alike (0/O/o, 1/I/l, 5/S/s, C/c, …).
* **Fonts (self-made):** all 62 characters rendered from 16 computer fonts at 2 sizes, drawn
  in FEMNIST's format (dark ink on white, 28×28, centered). The models never saw printed
  characters.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

HARD_GROUPS = ["0Oo", "1Il", "2Zz", "5Ss", "9gq", "Cc", "Kk", "Mm", "Pp", "Uu", "Vv", "Ww", "Xx", "Yy"]
HARD = sorted({LABELS.index(ch) for g in HARD_GROUPS for ch in g})
FONTS = ["DejaVuSans", "DejaVuSerif", "DejaVuSansMono", "LiberationSans-Regular", "LiberationSerif-Regular",
         "LiberationMono-Regular", "FreeSans", "FreeSerif", "FreeMono", "Carlito-Regular", "Caladea-Regular",
         "Inter-Regular", "JetBrainsMono-Regular", "FreeSansOblique", "Carlito-Italic", "Caladea-Italic"]
# on Windows the Linux fonts above are missing; these are used instead (results then differ slightly)
WINDOWS_FONTS = ["arial", "times", "cour", "calibri", "cambria", "georgia", "verdana", "tahoma",
                 "consola", "segoeui", "trebuc", "arialbd", "ariali", "timesi", "calibrii", "georgiai"]


def render_character(ch, font_path, size_px, ink=0.16):
    font = ImageFont.truetype(font_path, 200)
    img = Image.new("L", (280, 280), 0)
    ImageDraw.Draw(img).text((140, 140), ch, font=font, fill=255, anchor="mm")
    glyph = img.crop(img.getbbox())
    scale = size_px * 10 / max(glyph.size)
    glyph = glyph.resize((max(1, round(glyph.width * scale)), max(1, round(glyph.height * scale))), Image.LANCZOS)
    canvas = Image.new("L", (280, 280), 0)
    canvas.paste(glyph, ((280 - glyph.width) // 2, (280 - glyph.height) // 2))
    small = np.asarray(canvas.resize((28, 28), Image.BOX), dtype=np.float32) / 255.0
    return 1.0 - small * (1.0 - ink)


font_images, font_labels = [], []
for name in FONTS + WINDOWS_FONTS:
    hits = glob.glob("/usr/share/fonts/**/{}.[ot]tf".format(name), recursive=True) + \
        glob.glob("C:/Windows/Fonts/{}.ttf".format(name))
    if not hits:
        continue
    for size in [8, 11]:
        for y, ch in enumerate(LABELS):
            font_images.append(render_character(ch, hits[0], size)[None])
            font_labels.append(y)

test_sets = {
    "FEMNIST easy": (X_test_t[~np.isin(Y_test_np, HARD)], Y_test_np[~np.isin(Y_test_np, HARD)]),
    "FEMNIST hard": (X_test_t[np.isin(Y_test_np, HARD)], Y_test_np[np.isin(Y_test_np, HARD)]),
}
if font_images:
    test_sets["Fonts (self-made)"] = (torch.from_numpy(np.stack(font_images)), np.array(font_labels))
for k, (X, Y) in test_sets.items():
    print("{:18s} {:5d} images, {} classes".format(k, len(Y), len(set(Y.tolist()))))

if font_images:
    fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
    for ax, ch in zip(axes, "0OAh1lS5"):
        ax.imshow(font_images[LABELS.index(ch) + NUM_CLASSES * 3][0], cmap="gray", vmin=0, vmax=1)
        ax.set_title("'{}'".format(ch))
        ax.axis("off")
    plt.suptitle("Self-made font test images")
    plt.show()

In [ ]:
compare_rows = []
for model_name, m in models.items():
    for set_name, (X, Y) in test_sets.items():
        df, _ = extract_predictions(m, X, Y, ZERO)
        row = {"model": model_name, "test set": set_name, "accuracy": 100 * df.Correct.mean(),
               "top-1 confidence": df.Top1_Confidence.mean()}
        for ch in ["O", "A"]:
            mask = Y == LABELS.index(ch)
            row["'{}' recall".format(ch)] = 100 * (df[mask].Predicted_Label == ch).mean() if mask.any() else np.nan
        compare_rows.append(row)
compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
print(compare_df.round(2).to_string(index=False))

In [ ]:
pivot = compare_df.pivot(index="test set", columns="model", values="accuracy")
pivot.plot(kind="bar", figsize=(10, 5), rot=0)
plt.ylabel("Top-1 accuracy (%)")
plt.title("Models on easy, hard and self-made test data")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 15. Speed: latency and throughput

In [ ]:
m = baseline["model"]
with torch.no_grad():
    for _ in range(20):
        m(X_test_t[:1])
    times = []
    for i in range(500):
        t0 = time.perf_counter()
        m(X_test_t[i:i + 1])
        times.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    for i in range(0, 8320, 64):
        m(X_test_t[i:i + 64])
    throughput = 8320 / (time.perf_counter() - t0)

speed = pd.DataFrame([
    {"measure": "latency, 1 image (median)", "value": "{:.3f} ms".format(1000 * np.median(times))},
    {"measure": "latency, 1 image (p95)", "value": "{:.3f} ms".format(1000 * np.percentile(times, 95))},
    {"measure": "throughput, batch 64", "value": "{:,.0f} images/s".format(throughput)},
    {"measure": "one FL round (5 local epochs + test)", "value": "{:.2f} s".format(np.median(baseline["history"]["seconds"]))},
    {"measure": "full run, {} rounds".format(NUM_ROUNDS), "value": "{:.1f} min".format(sum(baseline["history"]["seconds"]) / 60)},
])
print("CPU, {} thread(s)".format(NUM_THREADS))
print(speed.to_string(index=False))

## 16. Summary

In [ ]:
summary = pd.DataFrame([
    {"finding": "'O' recall at 50% malicious (clean -> attacked)",
     "value": "{:.1f}% -> {:.1f}%".format(*grid_df[grid_df.pair == "O->0"].set_index("malicious %").loc[[0, 50], "source recall"])},
    {"finding": "'A' recall at 50% malicious (clean -> attacked)",
     "value": "{:.1f}% -> {:.1f}%".format(*grid_df[grid_df.pair == "A->h"].set_index("malicious %").loc[[0, 50], "source recall"])},
    {"finding": "global accuracy drop at 50% (O->0 / A->h)",
     "value": "{:.2f} / {:.2f} pts".format(*[grid_df[(grid_df.pair == p) & (grid_df["malicious %"] == 0)].accuracy.iloc[0]
                                            - grid_df[(grid_df.pair == p) & (grid_df["malicious %"] == 50)].accuracy.iloc[0]
                                            for p in ["O->0", "A->h"]])},
    {"finding": "smallest level with a clear effect", "value": "20% malicious (1-2 of 50 do nothing measurable)"},
    {"finding": "defense, damaging levels (10-20%)",
     "value": ", ".join("{} {}%: {}".format(r.pair, r["malicious %"], r["attackers caught"])
                        for _, r in defense_df[(defense_df.window == "rounds 10-200") & (defense_df["malicious %"] >= 10)].iterrows())},
])
print(summary.to_string(index=False))

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "femnist_label_flipping_results.xlsx")) as writer:
    baseline_df.to_excel(writer, sheet_name="Baseline_History", index=False)
    grid_df.to_excel(writer, sheet_name="Attack_Levels", index=False)
    pd.DataFrame(where_rows).to_excel(writer, sheet_name="Where_Images_Go", index=False)
    defense_df.to_excel(writer, sheet_name="Defense", index=False)
    confidence_df.to_excel(writer, sheet_name="Confidence", index=False)
    sample_comparison.to_excel(writer, sheet_name="O_Sample_Comparison", index=False)
    compare_df.to_excel(writer, sheet_name="Model_Comparison", index=False)
    speed.to_excel(writer, sheet_name="Speed", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)
print("Results saved to", os.path.join(OUTPUT_DIR, "femnist_label_flipping_results.xlsx"))

**Conclusions**

* **The attack is targeted.** At 50% malicious, `O` recall falls from 75% to 7% and `A`
  recall from 92% to 31%. The other classes change only by run-to-run noise.
* **It needs a crowd.** 1–2 malicious participants out of 50 have no measurable effect, and
  the damage is clear only from 20%. Whether the letters look alike matters only at the
  extremes.
* **Accuracy cannot see it.** Global accuracy drops by at most 1.9 points, about the attacked
  letter's share of the test set times the recall it lost. On FEMNIST no single-letter attack
  can cost more than about 5 points.
* **It is detectable.** Looking only at the attacked letter's output weights in each update,
  our participant rule flags every attacker whenever there are enough to do damage, and never
  flags anyone in a clean run.
* **It carries over.** The attacked models also mislabel printed characters, and they are
  confidently wrong rather than uncertain.